### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [3]:
## Open AI and Open Source model-- llama3, gemma2, mistral, etc.
import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [7]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x114f01f60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1147956c0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from langchain_core.messages import HumanMessage, SystemMessage

messages=[
    SystemMessage(content="Translate the following English text to Gujarati."),
    HumanMessage(content="Hello, how are you?")
]

result=model.invoke(messages)

In [11]:
result

AIMessage(content='નમસ્કાર, તમે શ્રીમંત છો?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 50, 'total_tokens': 92, 'completion_time': 0.068449863, 'completion_tokens_details': None, 'prompt_time': 0.002791256, 'prompt_tokens_details': None, 'queue_time': 0.009707618, 'total_time': 0.071241119}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6c980774ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b4e49-395d-7383-8ef3-36b3ba317a39-0', usage_metadata={'input_tokens': 50, 'output_tokens': 42, 'total_tokens': 92})

In [12]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'નમસ્કાર, તમે શ્રીમંત છો?'

In [14]:
## Using LCEL - chain the components
chain = model | parser
chain.invoke(messages)

'નમસ્કાર, તમે શ્રીમંત છો?'

In [15]:
## Prompt templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}:"

prompt= ChatPromptTemplate.from_messages(
    [("system", generic_template),("user", "{text}")]
)

In [17]:
result=prompt.invoke({"language":"Gujarati","text":"Hello, how are you?"})

In [18]:
result.to_messages()

[SystemMessage(content='Translate the following into Gujarati:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]

In [19]:
chain = prompt | model | parser
chain.invoke({"language":"Gujarati","text":"Hello, how are you?"})

'નમસ્તે, શું છે તમે?'